# 🎓 Intelligent Exam Question Analysis — Milestone 1
## Classical ML Pipeline for Exam Question Difficulty Prediction

### 📌 Objective
Design a machine learning–based analytics system that:
- Accepts exam question text
- Accepts student response data
- Extracts text and numeric features
- Predicts question difficulty (**Easy / Medium / Hard**)
- Demonstrates offline evaluation

### 📊 Dataset
We use the **SciQ dataset** (~13,679 science multiple-choice questions).

### ⚠️ Academic Transparency Notes
> **1. Difficulty labels are DERIVED, not ground truth.**
> **2. Student scores are SIMULATED.**
> **3. No LLMs are used.**

---
## 📦 Step 1 — Import Libraries

In [ ]:
import json
import os
import numpy as np
import pandas as pd
from scipy.sparse import hstack, csr_matrix

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix, classification_report

import warnings
warnings.filterwarnings('ignore')

---
## 📂 Step 2 — Dataset Loading

In [ ]:
DATA_DIR = os.path.join("backend", "data", "SciQ")

def load_sciq_split(filepath):
    with open(filepath, "r") as f:
        data = json.load(f)
    
    df = pd.DataFrame(data)
    df["combined_text"] = (
        df["question"].fillna("")
        + " " + df["correct_answer"].fillna("")
        + " " + df["distractor1"].fillna("")
        + " " + df["distractor2"].fillna("")
        + " " + df["distractor3"].fillna("")
        + " " + df["support"].fillna("")
    )
    return df

train_df = load_sciq_split(os.path.join(DATA_DIR, "train.json"))
valid_df = load_sciq_split(os.path.join(DATA_DIR, "valid.json"))
test_df  = load_sciq_split(os.path.join(DATA_DIR, "test.json"))

print(f"Train: {len(train_df)} | Valid: {len(valid_df)} | Test: {len(test_df)}")

---
## 🧹 Step 3 — Text Preprocessing Preview

In [ ]:
sample = train_df.iloc[0]
print(f"Question: {sample['question']}")
print(f"Combined Text Preview: {sample['combined_text'][:100]}...")

---
## 🏷️ Step 4 — Difficulty Label Assignment

In [ ]:
DIFFICULTY_DISTRIBUTION = {"Easy": 0.35, "Medium": 0.40, "Hard": 0.25}

def assign_difficulty_labels(df, seed=42):
    rng = np.random.RandomState(seed)
    n = len(df)
    labels = (
        ["Easy"] * int(n * DIFFICULTY_DISTRIBUTION["Easy"])
        + ["Medium"] * int(n * DIFFICULTY_DISTRIBUTION["Medium"])
        + ["Hard"] * (n - int(n * DIFFICULTY_DISTRIBUTION["Easy"]) - int(n * DIFFICULTY_DISTRIBUTION["Medium"]))
    )
    rng.shuffle(labels)
    df = df.copy()
    df["difficulty"] = labels
    return df

train_df = assign_difficulty_labels(train_df, seed=42)
valid_df = assign_difficulty_labels(valid_df, seed=43)
test_df  = assign_difficulty_labels(test_df,  seed=44)

print(train_df["difficulty"].value_counts())

---
## 🧪 Step 5 — Student Score Simulation

In [ ]:
SCORE_PARAMS = {
    "Easy":   {"mean": 72, "std": 15, "n_students": 15},
    "Medium": {"mean": 55, "std": 17, "n_students": 15},
    "Hard":   {"mean": 40, "std": 16, "n_students": 15},
}

def simulate_student_scores(df, seed=42):
    rng = np.random.RandomState(seed)
    df = df.copy()
    score_strings = []
    for _, row in df.iterrows():
        params = SCORE_PARAMS[row["difficulty"]]
        scores = rng.normal(params["mean"], params["std"], params["n_students"])
        scores = np.clip(scores, 0, 100).round(1)
        score_strings.append(",".join(str(s) for s in scores))
    df["student_scores"] = score_strings
    return df

train_df = simulate_student_scores(train_df, seed=42)
valid_df = simulate_student_scores(valid_df, seed=43)
test_df  = simulate_student_scores(test_df,  seed=44)

print(f"Simulated scores for {len(train_df)} training rows.")

---
## ⚙️ Step 6 — Feature Engineering

In [ ]:
PASS_THRESHOLD = 50

def compute_numeric_features(scores_series):
    rows = []
    for scores_str in scores_series:
        scores = np.array([float(s) for s in scores_str.split(",")])
        avg = float(np.mean(scores))
        var = float(np.var(scores))
        pr  = float(np.sum(scores >= PASS_THRESHOLD) / len(scores) * 100)
        rows.append({"avg_score": avg, "variance": var, "pass_rate": pr})
    return pd.DataFrame(rows)

def build_features(df, tfidf=None, scaler=None, fit=True):
    if fit:
        tfidf = TfidfVectorizer(max_features=5000, stop_words="english")
        X_text = tfidf.fit_transform(df["combined_text"])
    else:
        X_text = tfidf.transform(df["combined_text"])
    
    numeric_df = compute_numeric_features(df["student_scores"])
    if fit:
        scaler = StandardScaler()
        X_num = scaler.fit_transform(numeric_df)
    else:
        X_num = scaler.transform(numeric_df)
    
    X = hstack([X_text, csr_matrix(X_num)])
    return X, tfidf, scaler

X_train, tfidf, scaler = build_features(train_df, fit=True)
le = LabelEncoder()
y_train = le.fit_transform(train_df["difficulty"])

X_valid, _, _ = build_features(valid_df, tfidf=tfidf, scaler=scaler, fit=False)
X_test,  _, _ = build_features(test_df,  tfidf=tfidf, scaler=scaler, fit=False)
y_valid = le.transform(valid_df["difficulty"])
y_test  = le.transform(test_df["difficulty"])

print(f"Feature matrix shape: {X_train.shape}")

---
## 🤖 Step 7 — Model Training

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42, solver="lbfgs"),
    "Decision Tree": DecisionTreeClassifier(max_depth=10, random_state=42),
}

for name, model in models.items():
    model.fit(X_train, y_train)
    acc = accuracy_score(y_train, model.predict(X_train))
    print(f"{name} Train Accuracy: {acc:.4f}")

---
## 📈 Step 8 — Model Evaluation

In [ ]:
for name, model in models.items():
    y_pred = model.predict(X_test)
    acc  = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average="weighted", zero_division=0)
    rec  = recall_score(y_test, y_pred, average="weighted", zero_division=0)
    print(f"{name} -> Acc: {acc:.4f} | Prec: {prec:.4f} | Rec: {rec:.4f}")

---
## 🔮 Step 9 — Sample Predictions & Conceptual Observation

> **"Difficulty is defined primarily by how students perform, not by how a question reads."**

When the text input is empty, the numeric performance features completely dominate the prediction.

In [ ]:
def predict(text, scores, model):
    df = pd.DataFrame([{"combined_text": text, "student_scores": scores}])
    X, _, _ = build_features(df, tfidf=tfidf, scaler=scaler, fit=False)
    return le.inverse_transform([model.predict(X)[0]])[0]

q_full = "Explain the process of photosynthesis."
s_low = "35,28,42,20,30,25,38,22,40,18"

print(f"Full text + low scores  -> {predict(q_full, s_low, models['Logistic Regression'])}")
print(f"Empty text + low scores -> {predict('', s_low, models['Logistic Regression'])}")